# PsychoPy CSV Files

PsychoPy records eye-tracking samples through its ioHub framework. ioHub writes the
samples of every supported eye tracker into the same table layout, so a recording from a
Pupil Core looks the same as one from any other tracker. The example in this guide comes
from a Pupil Core recording made with PsychoPy.

ioHub stores the recording in an HDF5 file next to the other output of the experiment. The
gaze samples sit in the eye sample table of that file,
`/data_collection/events/eyetracker/BinocularEyeSampleEvent` for a binocular tracker and
`/data_collection/events/eyetracker/MonocularEyeSampleEvent` for a monocular one. These lines
export the binocular table to CSV with pandas:

```python
import pandas as pd

samples = pd.read_hdf(
    'experiment.hdf5',
    '/data_collection/events/eyetracker/BinocularEyeSampleEvent',
)
samples.to_csv('experiment.csv', index=False)
```

The export needs the `tables` package, which *pymovements* does not install. The exported CSV
contains columns like these:

| Column | Content |
|--------|---------|
| `device_time` | Timestamp from the eye tracker clock, in seconds |
| `left_gaze_x`, `left_gaze_y` | Gaze position of the left eye, in PsychoPy window units |
| `right_gaze_x`, `right_gaze_y` | Gaze position of the right eye, in PsychoPy window units |

The example file holds exactly these five columns. A full export carries further columns,
for example `time`, `logged_time` or pupil measures. They do not get in the way, as
{py:func}`~pymovements.gaze.from_csv` only reads the columns you name. The column names are
fixed by ioHub, not by your experiment. What you choose in PsychoPy is the window unit of
the gaze coordinates, see below.

No dedicated loading function is needed. A PsychoPy CSV loads with
{py:func}`~pymovements.gaze.from_csv`, like any other CSV file. Three settings differ
from the defaults described in the {doc}`CSV guide <csv>`.


In [ ]:
import polars as pl

import pymovements as pm

csv_example = pl.read_csv('../../examples/gaze-psychopy-example.csv')
csv_example.head(5)

## Time Unit

ioHub timestamps are in seconds, while {py:func}`~pymovements.gaze.from_csv` assumes
milliseconds by default. Pass `time_unit='s'`.

`device_time` counts from the tracker clock, not from the start of your experiment. The
timestamps are therefore large absolute values. This does not affect any processing step.

ioHub also writes a `time` column in seconds, on the ioHub clock that all ioHub events share.
Pass `time_column='time'` instead to align the samples with other ioHub events, such as
messages.


## Coordinate Origin

PsychoPy places the origin of its coordinate system at the center of the window. Most
eye-tracker exports place it in the upper left corner, which is also what the other pages of
this guide use. Tell *pymovements* about the center origin by setting
`origin='center'` in the {py:class}`~pymovements.Experiment`. The negative x values in the
example file are a direct consequence of the center origin: the gaze was left of the screen
center.

Two further points follow from PsychoPy's coordinate system:

- The gaze columns are in the window units of the ioHub configuration. The example was
  recorded in `pix`. The section on other window units below shows how to load the other
  units.
- The PsychoPy y axis points upwards. *pymovements* keeps the direction of the input, and
  with an upper left origin that direction is downwards. Vertical positions and the vertical
  velocity component therefore have the opposite sign from recordings with an upper left
  origin. Velocity magnitudes and event detection are not affected.


## Experiment

The screen geometry, the eye-to-screen distance and the sampling rate are not part of the
CSV file. Define them in an {py:class}`~pymovements.Experiment`. The values below are
placeholders. Replace them with the values of your setup. The timestamps in the example
file are about 8 ms apart, which corresponds to a sampling rate of about 120 Hz. Use the
rate your tracker was configured with.


In [ ]:
experiment = pm.Experiment(
    screen_width_px=1920,
    screen_height_px=1080,
    screen_width_cm=53,
    screen_height_cm=30,
    distance_cm=60,
    origin='center',
    sampling_rate=120,
)

## Loading

The four gaze columns are passed as `pixel_columns`. Both eyes are loaded, so the resulting
`pixel` column holds four components.


In [ ]:
gaze = pm.gaze.from_csv(
    '../../examples/gaze-psychopy-example.csv',
    experiment=experiment,
    time_column='device_time',
    time_unit='s',
    pixel_columns=['left_gaze_x', 'left_gaze_y', 'right_gaze_x', 'right_gaze_y'],
)

gaze.samples.head(5)

From here on, the data goes through the usual pipeline. With the center origin set in the
experiment, {py:meth}`~pymovements.Gaze.pix2deg` converts the pixel coordinates to degrees
of visual angle correctly.


In [ ]:
gaze.pix2deg()
gaze.pos2vel()

gaze.samples.head(5)

## Other Window Units

PsychoPy offers further window units besides `pix`. Convert gaze columns in `norm`, `height`
or `cm` to pixels before loading. All units share the center origin, so the conversion is a
scaling. Columns in `deg` already hold degrees of visual angle. Pass them as
`position_columns` instead of `pixel_columns` and skip {py:meth}`~pymovements.Gaze.pix2deg`.

| Unit | Meaning | Conversion to pixels |
|------|---------|----------------------|
| `norm` | -1 to 1 on both axes | x times half the width, y times half the height, in pixels |
| `height` | the window height is 1 | x and y times the height in pixels |
| `cm` | centimeters on the screen | x and y times the pixels per centimeter of the screen |
| `deg` | degrees of visual angle | none, load as `position_columns` |

Width and height refer to the PsychoPy window. For a full screen window they are the screen
values of the {py:class}`~pymovements.Experiment`.

For illustration, the example file is first expressed in `height` units. This frame stands in
for a recording made in `height`.

In [ ]:
gaze_columns = ['left_gaze_x', 'left_gaze_y', 'right_gaze_x', 'right_gaze_y']

height_example = csv_example.with_columns(
    pl.col(gaze_columns) / experiment.screen.height_px,
)

height_example.head(5)

Multiplying the gaze columns by the screen height in pixels converts them to pixels. The
converted frame loads with {py:class}`~pymovements.Gaze` and the same arguments as
{py:func}`~pymovements.gaze.from_csv` above.

In [ ]:
pixel_example = height_example.with_columns(
    pl.col(gaze_columns) * experiment.screen.height_px,
)

gaze = pm.Gaze(
    samples=pixel_example,
    experiment=experiment,
    time_column='device_time',
    time_unit='s',
    pixel_columns=gaze_columns,
)

gaze.samples.head(5)

## Loading a Whole Dataset

For a folder of PsychoPy recordings, put the same settings into a
{py:class}`~pymovements.DatasetDefinition`: the `experiment` as above and the
{py:func}`~pymovements.gaze.from_csv` arguments as `load_kwargs` of the gaze resource. The
{doc}`local dataset tutorial <../../tutorials/local-dataset>` walks through this.
